In [33]:
import heapq
import numpy as np
from collections import Counter

# -------- Huffman Coding & RLE --------
class Node:
    def __init__(self, symbol=None, freq=0):
        self.symbol = symbol
        self.freq = freq
        self.left = None
        self.right = None

    def __lt__(self, other):
        return self.freq < other.freq

def build_tree(freq):
    heap = [Node(sym, f) for sym, f in freq.items()]
    heapq.heapify(heap)

    while len(heap) > 1:
        l = heapq.heappop(heap)
        r = heapq.heappop(heap)

        parent = Node(freq=l.freq + r.freq)
        parent.left = l
        parent.right = r

        heapq.heappush(heap, parent)

    return heap[0] if heap else Node()

def generate_codes(node, code="", codes=None):
    if codes is None:
        codes = {}

    if node.symbol is not None:
        codes[node.symbol] = code
        return codes

    if node.left:
        generate_codes(node.left, code + "0", codes)
    if node.right:
        generate_codes(node.right, code + "1", codes)

    return codes

def rle_encode(arr):
    if len(arr) == 0:
        return []
    encoded = []
    count = 1
    for i in range(1, len(arr)):
        if arr[i] == arr[i-1]:
            count += 1
        else:
            encoded.append((arr[i-1], count))
            count = 1
    encoded.append((arr[-1], count))
    return encoded

def huffman_decode(bitstring, reverse_codes):
    decoded = []
    current = ""
    for bit in bitstring:
        current += bit
        if current in reverse_codes:
            decoded.append(reverse_codes[current])
            current = ""
    return decoded

def rle_decode(rle):
    arr = []
    for val, count in rle:
        arr.append(np.repeat(val, count))
    if arr:
        return np.concatenate(arr)
    return np.array([])

In [34]:
import pywt

# -------- 2D Discrete Wavelet Transform (DWT) Using PyWavelets --------
WAVELET_NAME = 'bior4.4' # The CDF 9/7 wavelet used in actual JPEG2000 lossy compression

def dwt2_multilevel(img, levels=3):
    img = img.astype(np.float32)
    # Perform multilevel 2D DWT. 
    # Returns [LL, (HL_n, LH_n, HH_n), ... (HL_1, LH_1, HH_1)] where n is the highest level
    coeffs = pywt.wavedec2(img, WAVELET_NAME, level=levels, mode='symmetric')
    
    LL = coeffs[0]
    detail_coeffs = coeffs[1:]
    
    # Store original shapes for accurate reverse
    shapes = [c[0].shape for c in detail_coeffs]
    
    # Pywt natively returns coeffs from highest level to lowest (i.e. smallest to largest). 
    # To align with our loop structure below, let's just keep the detail_coeffs list.
    return LL, detail_coeffs, shapes

def idwt2_multilevel(LL, detail_coeffs, shapes):
    # Reconstruct the image
    coeffs = [LL] + detail_coeffs
    img = pywt.waverec2(coeffs, WAVELET_NAME, mode='symmetric')
    return img

In [35]:
# -------- JPEG2000 Simplified Pipeline --------

def j2k_quantize(LL, coeffs, Q):
    # Dead-zone scalar uniform quantization approach (simplified as division + round based on step size Q)
    q_LL = np.round(LL / Q).astype(np.int32)
    q_coeffs = []
    for HL, LH, HH in coeffs:
        q_coeffs.append((
            np.round(HL / Q).astype(np.int32),
            np.round(LH / Q).astype(np.int32),
            np.round(HH / Q).astype(np.int32)
        ))
    return q_LL, q_coeffs

def j2k_inverse_quantize(q_LL, q_coeffs, Q):
    LL = (q_LL * Q).astype(np.float32)
    coeffs = []
    for HL, LH, HH in q_coeffs:
        coeffs.append((
            (HL * Q).astype(np.float32),
            (LH * Q).astype(np.float32),
            (HH * Q).astype(np.float32)
        ))
    return LL, coeffs

def flatten_dwt(q_LL, q_coeffs):
    """Flattens out subbands. By scanning each subband independently, we group spatial zeros better."""
    flat = [q_LL.flatten()]
    for HL, LH, HH in q_coeffs:
        flat.extend([HL.flatten(), LH.flatten(), HH.flatten()])
    return np.concatenate(flat)

def unflatten_dwt(flat, LL_shape, coeff_shapes):
    """Refolds flattened streams into subbands matrices."""
    length = LL_shape[0] * LL_shape[1]
    q_LL = flat[:length].reshape(LL_shape)
    idx = length
    
    q_coeffs = []
    for shape in coeff_shapes:
        sz = shape[0] * shape[1]
        q_LH = flat[idx : idx+sz].reshape(shape)
        idx += sz
        q_HL = flat[idx : idx+sz].reshape(shape)
        idx += sz
        q_HH = flat[idx : idx+sz].reshape(shape)
        idx += sz
        q_coeffs.append((q_LH, q_HL, q_HH))
        
    return q_LL, q_coeffs

def j2k_encoding(q_LL, q_coeffs):
    # Note: Traditional j2k uses EBCOT and tier 1/2. We substitute with 
    # highly straightforward RLE into Huffman.
    flat_array = flatten_dwt(q_LL, q_coeffs)
    rle_encoded = rle_encode(flat_array)
    
    flat_huff = []
    for item in rle_encoded:
        flat_huff.extend(item)
        
    freq = Counter(flat_huff)
    root = build_tree(freq)
    codes = generate_codes(root)
    
    encoded_bitstring = "".join(codes[s] for s in flat_huff)
    return encoded_bitstring, codes

def j2k_decoding(encoded_bitstring, codes, LL_shape, coeff_shapes, Q):
    reverse_codes = {v: k for k, v in codes.items()}
    
    rle_flat = huffman_decode(encoded_bitstring, reverse_codes)
    rle_tuples = [(rle_flat[i], rle_flat[i+1]) for i in range(0, len(rle_flat), 2)]
    
    flat_array = rle_decode(rle_tuples)
    
    q_LL, q_coeffs = unflatten_dwt(flat_array, LL_shape, coeff_shapes)
    LL, coeffs = j2k_inverse_quantize(q_LL, q_coeffs, Q)
    
    return LL, coeffs

def j2k_pipeline(image, Q, levels=3):
    image_float = image.astype(np.float32) - 128.0
    
    LL, coeffs, shapes = dwt2_multilevel(image_float, levels)
    
    LL_shape = LL.shape
    coeff_shapes = [c[0].shape for c in coeffs]
    
    q_LL, q_coeffs = j2k_quantize(LL, coeffs, Q)
    
    encoded_bitstring, codes = j2k_encoding(q_LL, q_coeffs)
    
    rec_LL, rec_coeffs = j2k_decoding(encoded_bitstring, codes, LL_shape, coeff_shapes, Q)
    
    reconstructed = idwt2_multilevel(rec_LL, rec_coeffs, shapes)
    reconstructed = np.clip(reconstructed + 128.0, 0, 255).astype(np.uint8)
    
    return reconstructed, encoded_bitstring, (LL_shape, coeff_shapes)

In [36]:
# -------- Metrics --------
from sklearn.metrics import mean_squared_error
from skimage.metrics import peak_signal_noise_ratio
import math

def compute_metrics(original, reconstructed, encoded_bitstring):
    original = original.astype(np.float32)
    reconstructed = reconstructed.astype(np.float32)

    mse = mean_squared_error(original.flatten(), reconstructed.flatten())
    psnr = peak_signal_noise_ratio(original, reconstructed, data_range=255)

    H, W = original.shape
    original_bits = H * W * 8
    
    # Calculate bytes / bandwidth from pure bit length
    compressed_bits = len(encoded_bitstring)

    compression_ratio = original_bits / compressed_bits if compressed_bits > 0 else 0
    bpp = compressed_bits / (H * W)

    return {
        "MSE": mse,
        "PSNR": psnr,
        "Compression_Ratio": compression_ratio,
        "BPP": bpp
    }

def compute_entropy(encoded_bitstring):
    freq = Counter(encoded_bitstring)
    total = len(encoded_bitstring)
    if total == 0: return 0
    entropy = -sum((c/total) * math.log2(c/total) for c in freq.values())
    return entropy

# Loading Datasets
The datasets used are Kodak, McM and SIPI

In [37]:
import os
from PIL import Image

def load_images(folder_path):
    images = []
    
    for file in os.listdir(folder_path):
        if file.lower().endswith(('.png', '.tif', '.tiff')):
            path = os.path.join(folder_path, file)
            try:
                img = Image.open(path)
                images.append({
                    "image": img,
                    "mode": img.mode,   # 'RGB', 'L', etc.
                    "name": file
                })
            except:
                print(f"Skipping {file}")
                
    return images

# Selection Table
datasets = {
    "Kodak": "kodak_dataset",
    "McM": "McM_dataset",
    "SIPI": "sipi_dataset"
}

print("Datasets mapped.")

Datasets mapped.


In [38]:
import pandas as pd

# Quantization steps - Higher Q => More Compression
Q_factors = [2, 5, 10, 20, 40, 80, 120, 160]
LEVELS = 3 # Number of DWT passes

for dataset_name, path in datasets.items():
    print(f"\n{'='*40}")
    print(f" Processing Dataset: {dataset_name}")
    print(f"{'='*40}")
    images = load_images(path)
    
    results = []

    for Q in Q_factors:
    
        for idx, item in enumerate(images):
            img = item["image"]
            mode = item["mode"]
            name = item["name"]
            print(f"[{dataset_name}] Processing {name} with Q={Q}... {idx + 1} / {len(images)}")
            
            if mode == "L":
                original = np.array(img).astype(np.float32)
                
                reconstructed, encoded, _ = j2k_pipeline(original, Q, levels=LEVELS)
                metrics = compute_metrics(original, reconstructed, encoded)
    
            else:
                img = img.convert("RGB")
                original = np.array(img)
    
                img_ycbcr = img.convert("YCbCr")
                img_np = np.array(img_ycbcr).astype(np.float32)
    
                Y  = img_np[:, :, 0]
                Cb = img_np[:, :, 1]
                Cr = img_np[:, :, 2]
    
                # Heavy quantization on Chroma (Cb/Cr) channels
                Y_rec, enc_Y, _   = j2k_pipeline(Y, Q, levels=LEVELS)
                Cb_rec, enc_Cb, _ = j2k_pipeline(Cb, Q * 2, levels=LEVELS)
                Cr_rec, enc_Cr, _ = j2k_pipeline(Cr, Q * 2, levels=LEVELS)
    
                recon_ycbcr = np.stack([Y_rec, Cb_rec, Cr_rec], axis=2).astype(np.uint8)
                reconstructed_img = Image.fromarray(recon_ycbcr, mode="YCbCr").convert("RGB")
                reconstructed = np.array(reconstructed_img)
    
                encoded = enc_Y + enc_Cb + enc_Cr
    
                metrics = compute_metrics(
                    original.mean(axis=2),
                    reconstructed.mean(axis=2),
                    encoded
                )
    
            entropy = compute_entropy(encoded)
                
            results.append({
                "Q": Q,
                "PSNR": metrics["PSNR"],
                "CR": metrics["Compression_Ratio"],
                "BPP": metrics["BPP"],
                "MSE": metrics["MSE"],
                "Entropy": entropy
            })
    
    # Convert results to DataFrame
    df = pd.DataFrame(results)
    
    # Average Metrics
    df_avg = df.groupby("Q").mean(numeric_only=True).reset_index()
    
    print(f"\nAverage metrics for {dataset_name}:")
    print(df_avg)
    
    # Save Results
    file_name = f"{dataset_name}_jpeg2000_full_results.csv"
    df.to_csv(file_name, index=False)
    
    file_name_avg = f"{dataset_name}_jpeg2000_avg_results.csv"
    df_avg.to_csv(file_name_avg, index=False)
    
    print(f"Saved results to {file_name} and {file_name_avg}")


 Processing Dataset: Kodak
[Kodak] Processing kodim23.png with Q=2... 1 / 24
[Kodak] Processing kodim05.png with Q=2... 2 / 24
[Kodak] Processing kodim08.png with Q=2... 3 / 24
[Kodak] Processing kodim16.png with Q=2... 4 / 24
[Kodak] Processing kodim22.png with Q=2... 5 / 24
[Kodak] Processing kodim11.png with Q=2... 6 / 24
[Kodak] Processing kodim06.png with Q=2... 7 / 24
[Kodak] Processing kodim19.png with Q=2... 8 / 24
[Kodak] Processing kodim21.png with Q=2... 9 / 24
[Kodak] Processing kodim15.png with Q=2... 10 / 24
[Kodak] Processing kodim13.png with Q=2... 11 / 24
[Kodak] Processing kodim18.png with Q=2... 12 / 24
[Kodak] Processing kodim20.png with Q=2... 13 / 24
[Kodak] Processing kodim01.png with Q=2... 14 / 24
[Kodak] Processing kodim12.png with Q=2... 15 / 24
[Kodak] Processing kodim04.png with Q=2... 16 / 24
[Kodak] Processing kodim17.png with Q=2... 17 / 24
[Kodak] Processing kodim07.png with Q=2... 18 / 24
[Kodak] Processing kodim24.png with Q=2... 19 / 24
[Kodak] Proc